## Step 1 : Data Cleaning & Feature Engineering

<h3>Load data :</h3>

In [15]:
import pandas as pd

df = pd.read_csv('uncleaned_Dataset_1k.csv')

print(df.head())
    


   Unnamed: 0             Product_Name                  Category_Path  \
0           0      smartwatch series x       Wearables > Smartwatches   
1           1  Gaming Desktop RTX 4070        Tech > Audio > Speakers   
2           2  Budget Office Laptop 15          ELECTRONICS > LAPTOPS   
3           3   SLIM ULTRABOOK 14-INCH  Accessories > Adapters & Hubs   
4           4  Gaming Desktop RTX 4070       Wearables > Smartwatches   

     Brand_Name                                    Raw_Description  \
0     Clicky     Affordable office laptop with long battery eff...   
1        SoundX  Compact wireless earbuds with deep bass. <br>W...   
2        SoundX  Ultra thin lightweight laptop for everyday off...   
3      techcorp     Mechanical keyboard with tactile RGB switches.   
4        SoundX     Mechanical keyboard with tactile RGB switches.   

  Price_String  
0          831  
1     $1826.26  
2      $614.99  
3    $1,476.81  
4          NaN  


<h3> Clening data :</h3>

**1/ Drop junk columns :**

there is a column named "Unnamed: 0" in the data frame : \
-> it has been added in the csv file creation phase  \
-> it is considered as junk data (junk index column) or useless 

we do not need that column because :

* <b>Our content-based system only cares about meaningful product attributes</b> (Product Name, Category, Brand, Description, and Price) not product index.

* A random ID or a row number carries zero textual or semantic meaning, including it in vectorization or similarity math would only add noise and distort calculations.

* Pandas already assigns its own built-in row index ($0, 1, 2, 3\dots$) automatically when we load the dataset, making that column completely redundant.

In [19]:
# Drop junk index column (useless and redundant as pandas already has a built-in row index)
if 'Unnamed: 0' in df.columns: # this looks at the dataset's column names and asks: "Is there a column named 'Unnamed: 0'?"
    df = df.drop(columns=['Unnamed: 0']) # If the column exists, delete the 'Unnamed: 0' column from your DataFrame.

print(df.columns) # the updated data frame without that 'Unnamed: 0' column

Index(['Product_Name', 'Category_Path', 'Brand_Name', 'Raw_Description',
       'Price_String'],
      dtype='str')


**2/ Drop duplicates :**

In [21]:
df = df.drop_duplicates() # If it finds 2 or 3 rows that share the exact same title, category, brand, description, and price, it keeps only the first instance and deletes the extra identical rows.

print(df)

                         Product_Name                  Category_Path  \
0                 smartwatch series x       Wearables > Smartwatches   
1             Gaming Desktop RTX 4070        Tech > Audio > Speakers   
2             Budget Office Laptop 15          ELECTRONICS > LAPTOPS   
3              SLIM ULTRABOOK 14-INCH  Accessories > Adapters & Hubs   
4             Gaming Desktop RTX 4070       Wearables > Smartwatches   
...                               ...                            ...   
1010     probook ultra laptop 15-inch         Electronics > Desktops   
1011      4K Ultra HD Monitor 27-inch                            NaN   
1012     Workstation Laptop Pro 16          Wearables > Smartwatches   
1013        Studio Monitor Headphones        Tech > Audio > Speakers   
1014         Ergonomic Wireless Mouse        Tech > Audio > Speakers   

           Brand_Name                                    Raw_Description  \
0           Clicky     Affordable office laptop with long b

**3/ Fill empty values :**

* targeting strictely the text columns (Product_Name , Category_Path , Brand_Name and Raw_Description) and filled any Nan or empty values with empty strings "".

In [40]:
# fill empty values ONLY in text columns
text_cols = ['Product_Name', 'Category_Path' , 'Brand_Name' , 'Raw_Description']
df[text_cols] = df[text_cols].fillna('')

print(df)

                         Product_Name                  Category_Path  \
0                 smartwatch series x       Wearables > Smartwatches   
1             Gaming Desktop RTX 4070        Tech > Audio > Speakers   
2             Budget Office Laptop 15          ELECTRONICS > LAPTOPS   
3              SLIM ULTRABOOK 14-INCH  Accessories > Adapters & Hubs   
4             Gaming Desktop RTX 4070       Wearables > Smartwatches   
...                               ...                            ...   
1010     probook ultra laptop 15-inch         Electronics > Desktops   
1011      4K Ultra HD Monitor 27-inch                                  
1012     Workstation Laptop Pro 16          Wearables > Smartwatches   
1013        Studio Monitor Headphones        Tech > Audio > Speakers   
1014         Ergonomic Wireless Mouse        Tech > Audio > Speakers   

           Brand_Name                                    Raw_Description  \
0           Clicky     Affordable office laptop with long b

## Building the model:

**RECALL : the Content-Based Filtering method using CountVectorizer :**

* Step 1 : <b>Build the Vocabulary</b> (CountVectorizer scans the entire column/columns you want to use and sorts every unique feature(ex: genre) it finds in a list in alphabetical order and removes duplicates), that big master list is called the vocabulary.

* Step 2 : <b>Fill in the Numbers (0s and 1s)</b> , CountVectorizer goes row by row and places a 1 if the feature (ex: genre) is present, or a 0 if it's missing.

* Step 3 : <b>Output as Numerical Vectors</b>, for example : Each movie is now represented as a simple array of numbers (list)

* Step 4 : <b>Similarity & Recommendation:</b> the Cosine Similarity compares those numerical vectors and gives a score between $0.0$ and $1.0$. The closer to $1.0$, the higher up it goes on your recommendation list.

## STEP 1 + 2 : Build the Vocabulary list + Fill in the numbers (Vectorize)

In [45]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

category_matrix = CountVectorizer(lowercase=True).fit_transform(df['Category_Path']) # convert all category names into numbers and put them into a list (The master vocabulary list)

'''
 CountVectorizer extracted a total vocabulary of 17 unique words (This is the master vocabulary list) , this list is sorted alphabeticaly + there is no dublicates + indexed (each item has its index) :

    ['accessories', 'adapters', 'audio', 'computers', 'desktops', 'earbuds', 'electronics', 'headphones', 'hubs', 'keyboards', 'laptops', 'mouse', 'smartwatches', 'speakers', 'tech', 'wearables', 'wireless']

'''

print(category_matrix)


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2187 stored elements and shape (1000, 17)>
  Coords	Values
  (0, 15)	1
  (0, 12)	1
  (1, 14)	1
  (1, 2)	1
  (1, 13)	1
  (2, 6)	1
  (2, 10)	1
  (3, 0)	1
  (3, 1)	1
  (3, 8)	1
  (4, 15)	1
  (4, 12)	1
  (5, 6)	1
  (5, 10)	1
  (6, 2)	1
  (6, 7)	1
  (7, 0)	1
  (7, 11)	1
  (8, 6)	1
  (8, 10)	1
  (10, 0)	1
  (10, 11)	1
  (11, 0)	1
  (11, 1)	1
  (11, 8)	1
  :	:
  (989, 6)	1
  (989, 10)	1
  (989, 3)	1
  (990, 14)	1
  (990, 2)	1
  (990, 13)	1
  (991, 2)	1
  (991, 7)	1
  (992, 0)	1
  (992, 11)	1
  (993, 14)	1
  (993, 2)	1
  (993, 13)	1
  (994, 6)	1
  (994, 10)	1
  (995, 6)	1
  (995, 4)	1
  (997, 15)	1
  (997, 12)	1
  (998, 14)	1
  (998, 2)	1
  (998, 13)	1
  (999, 14)	1
  (999, 2)	1
  (999, 13)	1


**Explanation of How does the CountVectorizer converts all Category names to numbers 0s and 1s :**

the <b>fit_transform</b> function does all the heavy lifting in two steps combined into one function call:

* 1/ <b>fit()</b> : <b>Scans the entire Category_Path column and builds the master Vocabulary List</b> : ['accessories', 'adapters', 'audio', 'computers', 'desktops', 'earbuds', 'electronics' ..etc].

* 2/ <b>transform()</b>: Goes row by row (product by product) and turns each Category_Path's text into an array of $1$s and $0$s , by placing 1 if the category is present in the vocabulary list and 0 if it's missing.



**Explanation of the output after "print(category_list)" :**

When you print a sparse matrix directly in Python, SciPy uses the coordinate format (Row, Column) Count:

(Row, Column) : Value \
  (0, 15)    :    1   --> Product 0 has 1 instance of Vocabulary Word #15 ("wearables") \
  (0, 12)    :    1   --> Product 0 has 1 instance of Vocabulary Word #12 ("smartwatches")

## STEP 3 : Output as numerical values

**Visualization of the Category_matrix :**

In [43]:
# Convert the category_matrix to a dense DataFrame with column names , to see the actual matches of the Category_path with each product row
matrix_df = pd.DataFrame(
    category_matrix.toarray(), 
    columns=CountVectorizer(lowercase=True).fit(df['Category_Path']).get_feature_names_out()
)

# Display the first 5 rows
print(matrix_df.head())

   accessories  adapters  audio  computers  desktops  earbuds  electronics  \
0            0         0      0          0         0        0            0   
1            0         0      1          0         0        0            0   
2            0         0      0          0         0        0            1   
3            1         1      0          0         0        0            0   
4            0         0      0          0         0        0            0   

   headphones  hubs  keyboards  laptops  mouse  smartwatches  speakers  tech  \
0           0     0          0        0      0             1         0     0   
1           0     0          0        0      0             0         1     1   
2           0     0          0        1      0             0         0     0   
3           0     1          0        0      0             0         0     0   
4           0     0          0        0      0             1         0     0   

   wearables  wireless  
0          1         0  


**So , What is that Category_matrix ?**

It is the final result!. It is a large table of numbers (a matrix) where :

* <b>Rows</b> = <b>Every product</b> in the dataset.

* <b>Columns</b> = <b>Every unique Category</b> in the Vocabulary list [accessories', 'adapters', 'audio', ...etc].

* <b>Values</b> = <b>1</b> if the product belongs to that Categoty, <b>0</b> if the product doesn't.

## STEP 4 PART 1: COSINE SIMILARITY

**we're going to calculate the similarity score between every single product and other product in our dataset all at once.**

In [46]:
# Compute Cosine Similarity Matrix
cosine_sim = cosine_similarity(category_matrix, category_matrix)

print(cosine_sim)

[[1. 0. 0. ... 1. 0. 0.]
 [0. 1. 0. ... 0. 1. 1.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [1. 0. 0. ... 1. 0. 0.]
 [0. 1. 0. ... 0. 1. 1.]
 [0. 1. 0. ... 0. 1. 1.]]


**Why pass (category_matrix, category_matrix) twice?**

we are feeding the exact same table of prosuct numerical vectors into both arguments of the <code>cosine_similarity</code> function:

* Argument 1 (category_matrix): List of all products as Rows.

* Argument 2 (category_matrix): List of all products as Columns.

<code>cosine_similarity</code> compares product 0 against every product, then product 1 against every product, then product 2 against every product, and so on.

So : <b><code>cosine_sim</code> becomes a matrix where both the rows and columns represent your products and the cells in the middle represents the similarity scores between the products.</b>

## STEP 4 PART 2 : The Recommendation function

In [50]:
def rcommend_products(name, top_n=5):
    idx = df[df['Product_Name'] == name].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1],reverse=True)[1 : top_n+1]

    product_indices = [i[0] for i in sim_scores]

    return df['Product_Name'].iloc[product_indices]


print(rcommend_products('Gaming Desktop RTX 4070'))

13       Workstation Laptop Pro 16   
25             slim ultrabook 14-inch
27                usb-c multiport hub
32                SMARTWATCH SERIES X
46            Budget Office Laptop 15
Name: Product_Name, dtype: str
